# 02 — Feature search & edit (reliability)

**What you have:** a scorer and a dataset from notebook `01`.
**What you'll produce:** a set of SAE features responsible for the behavior, a PISCES edit that suppresses them, and an evaluation showing the behavior drops *without* breaking general ability — checked against a random-feature control and a tau/mu sweep.
**What success looks like:** target-bad-behavior falls well below the notebook-`01` baseline while a general-behavior check stays high, and your chosen features beat the random control.

**For every experiment, ask: Why are we doing it? What are we doing? What did we get?**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
# Import PISCES + student_utils from the shared cluster install by default;
# override with the PISCES_ROOT env var, or fall back to walking up to editor.py.
PISCES_ROOT = os.environ.get('PISCES_ROOT', '/home/morg/students/yoavgurarieh/pisces_students')
if not os.path.exists(os.path.join(PISCES_ROOT, 'editor.py')):
    _d = os.getcwd()
    while not os.path.exists(os.path.join(_d, 'editor.py')) and _d != os.path.dirname(_d):
        _d = os.path.dirname(_d)
    PISCES_ROOT = _d
os.chdir(PISCES_ROOT); sys.path.insert(0, PISCES_ROOT)
print('PISCES root:', PISCES_ROOT)

In [ ]:
from student_utils.model_loading import load_student_model
model, tm = load_student_model()  # google/gemma-2-2b-it

In [ ]:
from student_utils.feature_search import build_or_load_feature_catalog
catalog = build_or_load_feature_catalog(model=model)  # builds once if the cache is missing

## 1. Bring over your dataset and scorers from notebook 01
Notebooks don't share state, so re-create what you built in `01`: the eval DataFrame `df` (with a `'response'` column once you generate) and your `score_reliability`. You also need a small set of **general-behavior controls** and a scorer that checks the edit didn't break the model.

**STUDENT TODO:** paste your `df` builder and scorer from `01`, then fill in the general-behavior pieces below.

In [ ]:
# STUDENT TODO: recreate `df` (your eval data, with a 'prompt' column) and your
# `score_reliability` from notebook 01.
raise NotImplementedError('bring over df and your scorer from notebook 01')

In [ ]:
# General-behavior controls: ordinary prompts unrelated to the target behavior, used to
# check the edit doesn't damage normal ability.
general_prompts = [
    # STUDENT TODO: a handful of normal prompts (facts, reasoning, chit-chat).
]

def score_general_behavior(prompt, response):
    """Return a dict measuring whether the model still works normally.

    Must include numeric 'looks_ok' in {0.0, 1.0}: 1.0 when the response is
    coherent and on-task (not degenerate, not a spurious refusal), else 0.0.
    """
    raise NotImplementedError('define a general-behavior / breakage check')

## 2. Token-based feature search
The quickest way to find candidate features: look for SAE features whose top/bottom vocabulary tokens relate to the behavior. Each search token must be a **single model token** (usually with a leading space, e.g. ' definitely').

**STUDENT TODO:** choose probe tokens, run the search, and read the candidates — look at `top_tokens` and `matched_tokens` and judge which features are genuinely on-concept.

In [ ]:
from student_utils.feature_search import search_features_by_tokens, show_feature_candidates
search_tokens = [
    # STUDENT TODO: single tokens related to the behavior, e.g. ' definitely'
]
candidates = search_features_by_tokens(model, catalog, search_tokens, minmatch=1)
show_feature_candidates(candidates)

## 3. Choose your features
**STUDENT TODO:** build a feature set from the candidates: 3–10 features, each with `layer`, `feature_id`, `sign` (`-1` = suppress) and a short `why`. Start small; you can iterate.

In [ ]:
reliability_feature_set = {
    'name': 'reliability_feature_set',
    'description': 'STUDENT TODO: what behavior these features capture',
    'features': [
        # STUDENT TODO: {'layer': ..., 'feature_id': ..., 'sign': -1, 'why': '...'},
    ],
}
from student_utils.pisces_adapter import validate_feature_set
validate_feature_set(reliability_feature_set)  # checks structure; an empty list is allowed as a placeholder

## 4. Apply the edit and evaluate
Suppress your features and re-measure: target behavior should drop, general behavior should stay high. The edit auto-reverts when the `with` block ends.

`tau`/`mu` below are a starting point — you'll tune them in the sweep (step 7).

In [ ]:
from student_utils.pisces_adapter import temporary_pisces_edit
from student_utils.generation import generate_many
from student_utils.datasets import dataset_to_prompts, make_eval_dataframe
from student_utils.scoring import apply_scorer, summarize_scores
from IPython.display import display

edit_config = {'tau': 0.9, 'mu': 8.0, 'linscale': True, 'use_signs': False, 'description': 'v1'}
with temporary_pisces_edit(model, reliability_feature_set, edit_config):
    df['response_edited'] = generate_many(tm, dataset_to_prompts(df), max_new_tokens=120)
    general_resp = generate_many(tm, general_prompts, max_new_tokens=120)

print('target (edited):')
display(summarize_scores(apply_scorer(df.assign(response=df['response_edited']), score_reliability)))
print('general behavior (edited):')
gdf = make_eval_dataframe(general_prompts); gdf['response'] = general_resp
display(summarize_scores(apply_scorer(gdf, score_general_behavior)))

## 5. Random-feature control
Would random features at the same layers do just as well? If so, your selection isn't specific to the behavior. This is a key sanity check.

In [ ]:
from student_utils.pisces_adapter import make_random_feature_set_like
rand = make_random_feature_set_like(reliability_feature_set, seed=0)
with temporary_pisces_edit(model, rand, edit_config):
    rand_resp = generate_many(tm, dataset_to_prompts(df), max_new_tokens=120)
summarize_scores(apply_scorer(df.assign(response=rand_resp), score_reliability))

## 6. If token search isn't enough: contrastive search (CRISP)
Token search only finds features with obvious vocabulary. A stronger method is **contrastive search**: run the model on prompts where the behavior is present (target) vs absent (control), and find the features that fire much more on target than on control. This is the idea behind CRISP (Ashuach et al. 2026, arXiv:2508.13650).

You write the ranking yourself. `find_contrastive_features` collects per-feature activations and hands your `select_fn` a `merged` table (one row per feature, with target/control firing counts and activations); your function decides which features to keep. Read the paper for how to compare the two sides.

**STUDENT TODO:** define target/control prompt sets and implement `my_selection`.

In [ ]:
# Prompts where the behavior IS present (target) vs matched prompts where it is NOT (control).
target_prompts = [
    # STUDENT TODO
]
control_prompts = [
    # STUDENT TODO
]

In [ ]:
def my_selection(merged):
    """Your contrastive ranking: pick the features to suppress.

    `merged` has one row per (layer, feature_id) with columns:
        firing_count_target / firing_count_control
        frac_firing_target  / frac_firing_control
        sum_act_target      / sum_act_control
        mean_act_target     / mean_act_control
    Return a DataFrame of the selected rows including 'layer', 'feature_id' and
    'sign' (= -1 to suppress). Keep the frac_firing_* columns if you want to plot.
    Idea (see CRISP, arXiv:2508.13650): rank by how much more a feature fires on
    target than control, then keep the strongly-activated, target-specific ones.
    """
    raise NotImplementedError('implement your contrastive ranking (see the paper)')

In [ ]:
from student_utils.feature_search import find_contrastive_features, show_feature_candidates
contrastive = find_contrastive_features(target_prompts, control_prompts, model,
                                        catalog=catalog, select_fn=my_selection)
show_feature_candidates(contrastive)

In [ ]:
# Optional: visualise target vs control firing (upper-left points are target-specific).
from student_utils.reporting import plot_contrastive_scatter
plot_contrastive_scatter(contrastive)

## 7. Strength sweep (tau / mu)
Stronger edits suppress more but risk breaking general behavior. Sweep a few values and look at the trade-off between target reduction and general degradation.

**STUDENT TODO:** choose the `tau` and `mu` values to try.

In [ ]:
import pandas as pd
from student_utils.reporting import plot_tradeoff
rows = []
for tau in [ ]:        # STUDENT TODO: a few firing thresholds, e.g. 0.8 - 0.95
    for mu in [ ]:     # STUDENT TODO: a few edit strengths, e.g. 4 - 16
        cfg = {'tau': tau, 'mu': mu, 'linscale': True, 'use_signs': False, 'description': f'{tau}/{mu}'}
        with temporary_pisces_edit(model, reliability_feature_set, cfg):
            t_resp = generate_many(tm, dataset_to_prompts(df), max_new_tokens=100)
            g_resp = generate_many(tm, general_prompts, max_new_tokens=100)
        t = apply_scorer(df.assign(response=t_resp), score_reliability)
        gdf = make_eval_dataframe(general_prompts); gdf['response'] = g_resp
        g = apply_scorer(gdf, score_general_behavior)
        rows.append({'tau': tau, 'mu': mu,
                     'target_bad': t['target_bad_behavior'].mean(),
                     'general_ok': g['looks_ok'].mean()})
sweep = pd.DataFrame(rows); sweep

In [ ]:
plot_tradeoff(sweep, 'target_bad', 'general_ok')

## What you should have now / write-up
Your best feature set and edit config, evidence it beats the random-feature control, and a trade-off curve. Note what worked, what didn't, and what you'd try next.